In [33]:
import mlflow
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from sklearn import linear_model
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('SCD')


<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1779782786199, experiment_id='0', last_update_time=1779786251550, lifecycle_stage='active', name='SCD', tags={}, trace_location=None, workspace='default'>

In [48]:
#Import data and convert the variable types to the correct format for modeling
scd = pd.read_excel('full_data.xlsx')
scd.head(5)
#scd.info()
#scd.describe()





,DNA NUMBER,APOL1 GENOTYPE,HMOX1 ALLELE 1,HMOX1 ALLELE 2,HMOX1 GENOTYPE,HMOX1_code,GENDER,AGE(Years),ETHNIC Group,BP,...,VOC,Transfusion,Serum Creatinine,HEIGHT(cm),GFR,Fetal Hb,LDH,BILI INDIRECT,bili DIRECT,Bili totale
0,D017-819,WILD TYPE,29.0,39.0,NaN,0.0,F,4,MUNDIBU,90/50,...,1,3,0.6,94,64.703333,13.8,250.9,0.1,0.2,0.3
1,D017-820,WILD TYPE,23.0,38.0,NaN,0.0,M,7,MUMBEKU,94/64,...,1,6,0.7,107,63.130000,17.8,1166.0,2.3,0.1,2.4
2,D017-821,G1 HETEROZYGOUS,31.0,31.0,NaN,1.0,F,10,TANDU,112/85,...,3,10,0.8,128,66.080000,15.0,1055.0,1.5,0.3,1.8
3,D017-822,G1 HETEROZYGOUS,23.0,35.0,NaN,1.0,F,9,TANDU,104/65,...,3,5,0.4,130,134.230000,4.6,1484.0,2.9,0.4,3.3
4,D017-825,G1 HETEROZYGOUS,31.0,31.0,NaN,1.0,F,11,TANDU,108/54,...,3,10,0.6,115,79.158333,2.0,2440.0,3.4,0.0,3.4


In [49]:
# Assign HMOX1 genotype based on allele values
scd['HMOX1 GENOTYPE'] = np.where(
    (scd['HMOX1 ALLELE 1'] < 25) & (scd['HMOX1 ALLELE 2'] < 25),
    'SS',
    np.where(
        (scd['HMOX1 ALLELE 1'] > 25) & (scd['HMOX1 ALLELE 2'] < 25),
        'LS',
        np.where(
            (scd['HMOX1 ALLELE 1'] < 25) & (scd['HMOX1 ALLELE 2'] > 25),
            'SL',
            np.where(
                (scd['HMOX1 ALLELE 1'] > 25) & (scd['HMOX1 ALLELE 2'] > 25),
                'LL',
                None
            )
        )
    )
)

scd['HMOX1 GENOTYPE'].value_counts()
scd.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 31 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   DNA NUMBER        266 non-null    object 
 1   APOL1 GENOTYPE    266 non-null    object 
 2   HMOX1 ALLELE 1    265 non-null    float64
 3   HMOX1 ALLELE 2    265 non-null    float64
 4   HMOX1 GENOTYPE    251 non-null    object 
 5   HMOX1_code        265 non-null    float64
 6   GENDER            266 non-null    object 
 7   AGE(Years)        266 non-null    int64  
 8   ETHNIC Group      266 non-null    object 
 9   BP                266 non-null    object 
 10  SBP               266 non-null    int64  
 11  DBP               266 non-null    int64  
 12  WEIGHT/Kg         266 non-null    float64
 13  HEIGHT/m          266 non-null    float64
 14  BMI (Kg/m2)       266 non-null    float64
 15  prot              266 non-null    object 
 16  GB/Nitr           266 non-null    object 
 1

In [50]:
#Prepare data for linear regression and split into train and test sets
# Encode object/categorical columns before modeling
cat_cols = scd.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns to encode:', cat_cols)
if cat_cols:
    scd = pd.get_dummies(scd, columns=cat_cols, drop_first=True)
scd.info()
X = scd.drop(columns=["GFR"])
y = scd['GFR']
X_train_df, X_test_df, Y_train_df, Y_test_df = train_test_split(X, y, test_size=0.2, random_state=42)
# Fit the linear regression model
model = LinearRegression()
fitted_model = model.fit(X_train_df, Y_train_df)
# Access fitted model attributes
print("Fitted model object:", fitted_model)
print("Coefficients:", fitted_model.coef_)
print("Intercept:", fitted_model.intercept_)
print("Train R^2:", fitted_model.score(X_train_df, Y_train_df))
print("Test R^2:", fitted_model.score(X_test_df, Y_test_df))

Categorical columns to encode: ['DNA NUMBER', 'APOL1 GENOTYPE', 'HMOX1 GENOTYPE', 'GENDER', 'ETHNIC Group', 'BP', 'prot', 'GB/Nitr', 'ACR', 'Alb/mg/l', 'Creatinine']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Columns: 1015 entries, HMOX1 ALLELE 1 to Creatinine_> 405,4
dtypes: bool(995), float64(13), int64(7)
memory usage: 300.2 KB


ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [2]:
with mlflow.start_run():
    mlflow.set_tag('developer','Oluyomi')
    mlflow.log_param('data-path', '908D4730.xlsx')

NameError: name 'mlflow' is not defined